## Assignment 5

In this assignment, I extend the self-attention classifier from the course material into a simplified Transformer encoder block by adding a feedforward network, two residual connections and two layer normalization layers. The model is trained and tested on the **IMDB movie review dataset**.

In [23]:
import keras
from keras.datasets import imdb

from keras.layers import (Input, Dense, Dropout, GlobalAveragePooling1D,
                          MultiHeadAttention, LayerNormalization, Add)

In [24]:
# Vocabulary size
max_features = 10000
# Words per sample
max_len = 250

In [25]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

In [26]:
# Pad all the samples to same length
x_train = keras.utils.pad_sequences(x_train, maxlen=max_len, padding='post')
x_test = keras.utils.pad_sequences(x_test, maxlen=max_len, padding='post')

The downloaded samples have already been tokenized, and converted to integer lists of token indices. They have been padded to a common length of 250 tokens.

We now build dictionaries to convert the integer indices in the sample lists to the corresponding words and back. 

The first three indices have special meanings (0: padding, 1: sequence start, 2: unknown word).

In [27]:
word_to_index = imdb.get_word_index()
index_to_word = dict((value, key) for (key, value) in word_to_index.items())

example_review = " ".join(index_to_word.get(i-3, "?") for i in x_train[0])
print(example_review)

? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't you thi

## Token and positional embedding

The integer sequences are converted into word vectors. A positional embedding is added to the token embedding so the model can use word order.

In [28]:
from keras.layers import Layer, Embedding

class TokenAndPositionEmbedding(Layer):
    def __init__(self, seq_len, vocab_size, emb_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=emb_dim)
        self.pos_emb = Embedding(input_dim=seq_len, output_dim=emb_dim)


    def call(self, x_input):
        seq_len = keras.ops.shape(x_input)[-1]
        positions = keras.ops.arange(start=0, stop=seq_len, step=1)
        positions = self.pos_emb(positions)
        x_input = self.token_emb(x_input)
        return x_input + positions


## Baseline Model

**Classification model** with the new order-sensitive embedding layer is created.
The pooling layer is added to flatten the vectorized sequences. We also include a **Dropout layer** to reduce overfitting.

In [29]:
from keras.layers import Input, Dense, GlobalAveragePooling1D, Dropout
from keras.models import Model

In [30]:
# Word embeddings dimension
embed_dim = 32

inputs = Input(shape=(max_len,))
x = TokenAndPositionEmbedding(max_len, max_features, embed_dim)(inputs) 
x = GlobalAveragePooling1D()(x) 
x = Dropout(0.5)(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs, outputs=outputs) 

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
    )

model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 250)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_4  │ (None, 250, 32)        │       328,000 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_3      │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 328,033 (1.25 MB)

 Trainable params: 328,033 (1.25 MB)

 Non-trainable params: 0 (0.00 B)

Now train the model

In [31]:
model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=32
)

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - accuracy: 0.6608 - loss: 0.6459
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.8064 - loss: 0.4890
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8493 - loss: 0.3844
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8736 - loss: 0.3299
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8846 - loss: 0.2950


In [44]:
test_loss, test_acc = model.evaluate(x_test, y_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8760 - loss: 0.3102


In [45]:
print(f'Test accuracy = {test_acc:.4f}')
print(f'Test loss = {test_loss:.4f}')

Test accuracy = 0.8760
Test loss = 0.3102


### Baseline model results

| Epoch | Training accuracy | Training loss |
|---|---|---|
| 1 | 0.6608 | 0.6459 |
| 2 | 0.8064 | 0.4890 |
| 3 | 0.8493 | 0.3844 |
| 4 | 0.8736 | 0.3299 |
| 5 | 0.8846 | 0.2950 |

**Test accuracy: 0.8760**
**Test loss: 0.3102**

The **training accuracy** rose steadily from around **66,1%** to **88,5%** over five epochs.

The **training loss** dropped from around **0.646 to 0.295** meaning that the model was still improving when the training stopped.

The **test accuracy (87.6 %)** is little bit below the final **training accuracy (88.5 %)**. The gap suggests only a mild overfitting as it's quite small. Most likely the `Dropout layer` helped to limit it.

## Self-attention Model

A multi-head self-attention layer is added after the embedding layer. This is the starting point for the Transformer model.

In [33]:
from keras.layers import MultiHeadAttention

# Num of attention heads
num_heads = 2
# Query/key dimension for one head
key_dim = embed_dim // num_heads


inputs = Input(shape=(max_len,))
x = TokenAndPositionEmbedding(max_len, max_features, embed_dim)(inputs)
attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
x = GlobalAveragePooling1D()(attention_output) 
x = Dropout(0.5)(x)
outputs = Dense(1, activation='sigmoid')(x)
att_model = Model(
        inputs=inputs, 
        outputs=outputs
        )

att_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

att_model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 250)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_and_position… │ (None, 250, 32)   │    328,000 │ input_layer_3[0]… │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 250, 32)   │      4,224 │ token_and_positi… │
│ (MultiHeadAttentio… │                   │            │ token_and_positi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ multi_head_atten… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 32)        │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 1)         │         33 │ dropout_8[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 332,257 (1.27 MB)

 Trainable params: 332,257 (1.27 MB)

 Non-trainable params: 0 (0.00 B)

In [34]:
att_model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=32
)

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 33s 38ms/step - accuracy: 0.7863 - loss: 0.4334
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 37ms/step - accuracy: 0.9010 - loss: 0.2452
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 35ms/step - accuracy: 0.9326 - loss: 0.1829
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 29s 37ms/step - accuracy: 0.9476 - loss: 0.1413
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 36ms/step - accuracy: 0.9628 - loss: 0.1070


In [41]:
test_loss, test_acc = att_model.evaluate(x_test, y_test)

782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.8538 - loss: 0.4999


In [43]:
print(f'Test accuracy = {test_acc:.4f}')
print(f'Test loss = {test_loss:.4f}')

Test accuracy = 0.8538
Test loss = 0.4999


### Self-attention model results

| Epoch | Training accuracy | Training loss |
|---|---|---|
| 1 | 0.7863 | 0.4334 |
| 2 | 0.9010 | 0.2452 |
| 3 | 0.9326 | 0.1829 |
| 4 | 0.9476 | 0.1413 |
| 5 | 0.9628 | 0.1070 |

**Test accuracy: 85,38%.**

**Test loss: 0.4999.**

The **self-attention model** learned much faster than the **baseline model**. 

Training accuracy reached **78,6%** after the first epoch and **96,3%** after the fifth, while the training loss fell from 0.433 to 0.107. 

However, the test accuracy was only **85,4** %, which is about **11%** below the final training accuracy and also lower than the baseline's **87,6%**. The test loss **0,500** is much higher than the final training loss **0,107**.

The large gap in test loss indicates clear **overfitting**: the attention layer adds capacity that lets the model fit the training reviews very closely, but this does not generalize to unseen reviews. Within five epochs, the added complexity therefore did not improve on the simple baseline. 

Training was also considerably slower. With baseline model each epoch was around 4 seconds while in this model each epoch took around 30 seconds.

## Simplified Transformer encoder block

The attention model is expanded with the following components:

- **Residual connections (2):** the input of the attention layer is added to its output, and likewise for the feedforward network (`Add` layer). This helps gradients flow and lets each sublayer learn a correction to its input.
- **Layer normalization (2):** applied directly after each residual connection. It normalizes each token's feature vector, independent of batch statistics.
- **Feedforward network:** two Dense layers, the first with ReLU (64 units) and the second without activation.
- **Global average pooling and a sigmoid output layer** for binary classification.

**Tensor dimensions:** a residual connection adds two tensors element-wise, so their shapes must match. The embedding output has shape `(batch, 250, 32)`. `MultiHeadAttention` returns the same shape, and the second Dense layer projects the feedforward output back to `embed_dim = 32`, so both residual connections are valid.

In [36]:
# Embedding dimension
embed_dim = 32
# Attention heads
num_heads = 2
key_dim = embed_dim // num_heads
# Hidden units of the feedforward network
ff_dim = 64

In [37]:
x = TokenAndPositionEmbedding(max_len, max_features, embed_dim)(inputs)
attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
x = Add()([x, attention_output])
x = LayerNormalization(epsilon=1e-6)(x)
ff_output = Dense(ff_dim, activation='relu')(x)
ff_output = Dense(embed_dim)(ff_output)
x = Add()([x, ff_output])
x = LayerNormalization(epsilon=1e-6)(x)


x = GlobalAveragePooling1D()(x)
x = Dropout(0.5)(x)
outputs = Dense(1, activation='sigmoid')(x)

transformer_model = Model(inputs=inputs, outputs=outputs)

transformer_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

transformer_model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 250)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_and_position… │ (None, 250, 32)   │    328,000 │ input_layer_3[0]… │
│ (TokenAndPositionE… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 250, 32)   │      4,224 │ token_and_positi… │
│ (MultiHeadAttentio… │                   │            │ token_and_positi… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 250, 32)   │          0 │ token_and_positi… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 250, 32)   │         64 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 250, 64)   │      2,112 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 250, 32)   │      2,080 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 250, 32)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 250, 32)   │         64 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 32)        │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 32)        │          0 │ global_average_p… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 1)         │         33 │ dropout_10[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 336,577 (1.28 MB)

 Trainable params: 336,577 (1.28 MB)

 Non-trainable params: 0 (0.00 B)

### Training and testing

The model is trained for 5 epochs, using 20% of the training data for validation, and then evaluated on the test set.

In [38]:
transformer_model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

test_loss, test_acc = transformer_model.evaluate(x_test, y_test)
print(f'Test accuracy: {test_acc:.4f}')

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 33s 46ms/step - accuracy: 0.7734 - loss: 0.4511 - val_accuracy: 0.8782 - val_loss: 0.2905
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 30s 47ms/step - accuracy: 0.9110 - loss: 0.2280 - val_accuracy: 0.8764 - val_loss: 0.3009
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 28s 45ms/step - accuracy: 0.9422 - loss: 0.1622 - val_accuracy: 0.8790 - val_loss: 0.3056
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 29s 46ms/step - accuracy: 0.9609 - loss: 0.1181 - val_accuracy: 0.8700 - val_loss: 0.3573
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 28s 45ms/step - accuracy: 0.9729 - loss: 0.0868 - val_accuracy: 0.8704 - val_loss: 0.4606
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.8506 - loss: 0.5001
Test accuracy: 0.8506


In [39]:
print(f'Test loss: {test_loss:.4f}')

Test loss: 0.5001


### Simplified Transformer encoder results

| Epoch | Training accuracy | Training loss | Validation accuracy | Validation loss |
|---|---|---|---|---|
| 1 | 0.7734 | 0.4511 | 0.8782 | 0.2905 |
| 2 | 0.9110 | 0.2280 | 0.8764 | 0.3009 |
| 3 | 0.9422 | 0.1622 | 0.8790 | 0.3056 |
| 4 | 0.9609 | 0.1181 | 0.8700 | 0.3573 |
| 5 | 0.9729 | 0.0868 | 0.8704 | 0.4606 |

**Test accuracy: 85,06%**
**Test loss: 0.5001**

The Transformer model fit the training data quickly. 

**Training accuracy** rose from **77,3%** to **97,3%** and the **training loss** fell from **0,451** to **0,087**. 

The **validation** results tell a different story. **Validation accuracy** was already **87,8%** after the first epoch and stayed around **87-88%** for the rest of training, while the validation loss was lowest in **epoch 1 (0.291)** and rose every epoch after that, reaching **0.461** in **epoch 5**. 

This pattern shows clear overfitting from about **epoch 2-3** onward, so the model would probably have been best stopped after two or three epochs (for example using the `EarlyStopping` argument). 

The final test accuracy was 85,1% roughly 12 percentage points below the final training accuracy.